# Web Scraping with Requests & BeautifulSoup 🕷️
**Website:** https://quotes.toscrape.com

- `requests` library se HTML fetch karenge
- `BeautifulSoup` se data extract karenge
- **Saare pages** ka data ek saath collect karenge

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Step 1: Pehle sirf Page 1 ka HTML fetch karte hain

In [ ]:
url = "https://quotes.toscrape.com/page/1/"
response = requests.get(url)
print(f"Status Code: {response.status_code}")
print(f"Page Length: {len(response.text)} characters")

## Step 2: BeautifulSoup se HTML parse karte hain

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

# Ek quote extract karke dekhte hain
first_quote = soup.find("div", class_="quote")

text = first_quote.find("span", class_="text").get_text()
author = first_quote.find("small", class_="author").get_text()
tags = [tag.get_text() for tag in first_quote.find_all("a", class_="tag")]

print(f"Quote: {text}")
print(f"Author: {author}")
print(f"Tags: {tags}")

## Step 3: Page 1 ke SAARE quotes extract karte hain

In [ ]:
all_quotes = soup.find_all("div", class_="quote")
print(f"Page 1 par total quotes: {len(all_quotes)}")

for q in all_quotes:
    text = q.find("span", class_="text").get_text()
    author = q.find("small", class_="author").get_text()
    print(f"\n{author}: {text[:80]}...")

## Step 4: ⭐ SAARE PAGES se data extract karte hain (Loop)
Har page pe "Next" button check karenge. Jab tak next page milta rahega, loop chalega.

In [ ]:
base_url = "https://quotes.toscrape.com"
page_url = "/page/1/"

all_data = []  # Saara data yahan store hoga
page_number = 1

while page_url:
    # Page fetch karo
    response = requests.get(base_url + page_url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Saare quotes extract karo
    quotes = soup.find_all("div", class_="quote")
    
    for q in quotes:
        text = q.find("span", class_="text").get_text()
        author = q.find("small", class_="author").get_text()
        tags = [tag.get_text() for tag in q.find_all("a", class_="tag")]
        
        all_data.append({
            "Quote": text,
            "Author": author,
            "Tags": ", ".join(tags)
        })
    
    print(f"✅ Page {page_number} scraped - {len(quotes)} quotes found")
    
    # Next page check karo
    next_btn = soup.find("li", class_="next")
    if next_btn:
        page_url = next_btn.find("a")["href"]
        page_number += 1
    else:
        page_url = None  # Koi next page nahi, loop khatam
        print(f"\n🎉 Scraping complete! Total quotes collected: {len(all_data)}")

## Step 5: Data ko DataFrame mein convert karte hain

In [ ]:
df = pd.DataFrame(all_data)
print(f"Total Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
# Saara data dekhein
df

## Step 6: Data ko CSV file mein save karte hain

In [ ]:
df.to_csv("quotes_data.csv", index=False)
print("✅ Data saved to quotes_data.csv")